In [ ]:
!pip install -q -U ultralytics fiftyone

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [ ]:
import os, random, shutil
from pathlib import Path
from collections import Counter, defaultdict

OUT = Path("/content/ready_room")
SEED = 42
CAP = 1000          # max boxes per class
MIN_BOXES = 300     # warn below this
OI_PER_CLASS = 600
OI_BIG = 300        # big classes already plentiful in HomeObjects
MAX_SIDE = 1024     # resize on export to save disk

CLASSES = ["shelf", "wardrobe", "dresser", "refrigerator", "tv", "ceiling_fan",
           "ceiling_light", "wall_art", "window", "mirror", "table", "bed", "sofa", "door"]
CID = {c: i for i, c in enumerate(CLASSES)}
BIG = {"bed", "sofa", "table", "door", "window"}

HOME_MAP = {"bed": "bed", "sofa": "sofa", "table": "table", "tv": "tv",
            "wardrobe": "wardrobe", "window": "window", "door": "door",
            "photo frame": "wall_art"}   # chair, lamp, laptop, potted plant dropped

OI_MAP = {
    "Bookcase": "shelf", "Shelf": "shelf",
    "Chest of drawers": "dresser",
    "Cupboard": "wardrobe", "Wardrobe": "wardrobe",
    "Refrigerator": "refrigerator",
    "Television": "tv",
    "Ceiling fan": "ceiling_fan",
    "Chandelier": "ceiling_light",
    "Picture frame": "wall_art",
    "Mirror": "mirror", "Window": "window", "Door": "door",
    "Bed": "bed", "Couch": "sofa", "Studio couch": "sofa",
    "Table": "table", "Coffee table": "table", "Kitchen & dining room table": "table",
}

import fiftyone.utils.openimages as fouo
oi_all = set(fouo.get_classes(version="v7"))
print("NOT in Open Images (removed):", [k for k in OI_MAP if k not in oi_all])
OI_MAP = {k: v for k, v in OI_MAP.items() if k in oi_all}
for w in ["wardrobe", "cupboard", "cabinet", "drawer", "shelf", "fan", "light", "mirror", "frame"]:
    print(f"{w:9s} ->", sorted(c for c in oi_all if w in c.lower()))

/usr/local/lib/python3.13/dist-packages/glob2/fnmatch.py:141: SyntaxWarning: invalid escape sequence '\Z'
  return '(?ms)' + res + '\Z'


INFO:fiftyone.utils.openimages:Downloading 'https://storage.googleapis.com/openimages/v5/class-descriptions-boxable.csv' to '/tmp/tmpxu53k6yk/metadata/classes.csv'


NOT in Open Images (removed): ['Chandelier']
wardrobe  -> ['Wardrobe']
cupboard  -> ['Cupboard']
cabinet   -> ['Bathroom cabinet', 'Cabinetry', 'Filing cabinet']
drawer    -> ['Chest of drawers', 'Drawer']
shelf     -> ['Shelf']
fan       -> ['Ceiling fan', 'Infant bed', 'Mechanical fan']
light     -> ['Flashlight', 'Light bulb', 'Light switch', 'Lighthouse', 'Street light', 'Traffic light']
mirror    -> ['Mirror']
frame     -> ['Picture frame']


In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz

targets = defaultdict(list)
for oi, t in OI_MAP.items():
    targets[t].append(oi)

oi_datasets = []
for t, oi_names in targets.items():
    n = OI_BIG if t in BIG else OI_PER_CLASS
    print(f"\n=== {t} <- {oi_names} (max {n})")
    ds = foz.load_zoo_dataset(
        "open-images-v7", split="train", label_types=["detections"],
        classes=oi_names, max_samples=n, shuffle=True, seed=SEED,
        dataset_name=f"oi_{t}", drop_existing_dataset=True,
    )  # only_matching defaults to False -> all objects in the photo keep their labels
    oi_datasets.append(ds)


=== shelf <- ['Bookcase', 'Shelf'] (max 600)


INFO:fiftyone.zoo.datasets:Downloading split 'train' to '/root/fiftyone/open-images-v7/train' if necessary


INFO:fiftyone.utils.openimages:Downloading 'https://storage.googleapis.com/openimages/2018_04/train/train-images-boxable-with-rotation.csv' to '/root/fiftyone/open-images-v7/train/metadata/image_ids.csv'


 100% |██████|    4.8Gb/4.8Gb [15.7s elapsed, 0s remaining, 328.6Mb/s]      


INFO:eta.core.utils: 100% |██████|    4.8Gb/4.8Gb [15.7s elapsed, 0s remaining, 328.6Mb/s]      


INFO:fiftyone.utils.openimages:Downloading 'https://storage.googleapis.com/openimages/v5/class-descriptions-boxable.csv' to '/root/fiftyone/open-images-v7/train/metadata/classes.csv'


... (177 outputs left out)

 100% |█████████████████| 300/300 [4.1s elapsed, 0s remaining, 71.1 samples/s]      


INFO:eta.core.utils: 100% |█████████████████| 300/300 [4.1s elapsed, 0s remaining, 71.1 samples/s]      


Dataset 'oi_table' created


INFO:fiftyone.zoo.datasets:Dataset 'oi_table' created


In [ ]:
from ultralytics.data.utils import check_det_dataset, img2label_paths, IMG_FORMATS

def clean(c, xc, yc, w, h):
    x1, y1 = max(0.0, xc - w / 2), max(0.0, yc - h / 2)
    x2, y2 = min(1.0, xc + w / 2), min(1.0, yc + h / 2)
    if x2 - x1 < 1e-3 or y2 - y1 < 1e-3:
        return None
    return (c, (x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1)

pool = {}  # image path -> [(cls, xc, yc, w, h), ...]

# Open Images
group_drop = 0
for ds in oi_datasets:
    field = list(ds.get_field_schema(embedded_doc_type=fo.Detections))[0]
    for s in ds.iter_samples(progress=True):
        dets = s[field].detections if s[field] else []
        boxes, bad = [], False
        for d in dets:
            t = OI_MAP.get(d.label)
            if t is None:
                continue
            if getattr(d, "IsGroupOf", False):   # one box over many objects -> drop image
                bad = True
                break
            if getattr(d, "IsDepiction", False): # TV in a painting etc. -> drop box
                continue
            x, y, w, h = d.bounding_box          # FiftyOne: top-left x,y,w,h (relative)
            b = clean(CID[t], x + w / 2, y + h / 2, w, h)
            if b:
                boxes.append(b)
        if bad:
            group_drop += 1
            continue
        if boxes:
            pool[s.filepath] = boxes
print(f"Open Images: {len(pool)} images kept, {group_drop} dropped (group-of)")

# HomeObjects-3K
ho = check_det_dataset("HomeObjects-3K.yaml")   # auto-downloads
print("HomeObjects names:", ho["names"])
ho_map = {i: CID[HOME_MAP[n]] for i, n in ho["names"].items() if n in HOME_MAP}
n0 = len(pool)
for split in ("train", "val", "test"):
    dirs = ho.get(split)
    if not dirs:
        continue
    for d in (dirs if isinstance(dirs, list) else [dirs]):
        imgs = [str(p) for p in Path(d).rglob("*") if p.suffix[1:].lower() in IMG_FORMATS]
        for img, lbl in zip(imgs, img2label_paths(imgs)):
            if not os.path.exists(lbl):
                continue
            boxes = []
            for line in open(lbl):
                p = line.split()
                if len(p) == 5 and int(float(p[0])) in ho_map:
                    b = clean(ho_map[int(float(p[0]))], *map(float, p[1:]))
                    if b:
                        boxes.append(b)
            if boxes:
                pool[img] = boxes
print(f"HomeObjects: {len(pool) - n0} images kept | pool total: {len(pool)}")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
 100% |█████████████████| 600/600 [4.9s elapsed, 0s remaining, 119.6 samples/s]      


INFO:eta.core.utils: 100% |█████████████████| 600/600 [4.9s elapsed, 0s remaining, 119.6 samples/s]      


 100% |█████████████████| 600/600 [2.8s elapsed, 0s remaining, 245.8 samples/s]      


INFO:eta.core.utils: 100% |█████████████████| 600/600 [2.8s elapsed, 0s remaining, 245.8 samples/s]      


 100% |█████████████████| 600/600 [3.9s elapsed, 0s remaining, 144.5 samples/s]      


INFO:eta.core.utils: 100% |█████████████████| 600/600 [3.9s elapsed, 0s remaining, 144.5 samples/s]      


... (17 outputs left out)

INFO:eta.core.utils: 100% |█████████████████| 300/300 [2.1s elapsed, 0s remaining, 135.9 samples/s]      


 100% |█████████████████| 300/300 [2.4s elapsed, 0s remaining, 151.8 samples/s]      


INFO:eta.core.utils: 100% |█████████████████| 300/300 [2.4s elapsed, 0s remaining, 151.8 samples/s]      


Open Images: 4753 images kept, 372 dropped (group-of)

WARNING ⚠️ Dataset 'HomeObjects-3K.yaml' images not found, missing path '/content/datasets/homeobjects-3K/images/val'
Unzipping /content/datasets/homeobjects-3K.zip to /content/datasets/homeobjects-3K...: 100% ━━━━━━━━━━━━ 5386/5386 1.3Kfiles/s 4.1s
Dataset download success ✅ (17.7s), saved to /content/datasets

HomeObjects names: {0: 'bed', 1: 'sofa', 2: 'chair', 3: 'table', 4: 'lamp', 5: 'tv', 6: 'laptop', 7: 'wardrobe', 8: 'window', 9: 'door', 10: 'potted plant', 11: 'photo frame'}
HomeObjects: 2553 images kept | pool total: 7306


In [ ]:
from PIL import Image, ImageOps

def dhash(path, s=8):
    im = Image.open(path)
    im.draft("L", (64, 64))
    px = list(im.convert("L").resize((s + 1, s)).getdata())
    bits = 0
    for r in range(s):
        for c in range(s):
            bits = (bits << 1) | (px[r * (s + 1) + c] > px[r * (s + 1) + c + 1])
    return bits

seen, deduped, dup, broken = set(), {}, 0, 0
for img, boxes in pool.items():
    try:
        h = dhash(img)
    except Exception:
        broken += 1
        continue
    if h in seen:
        dup += 1
        continue
    seen.add(h)
    deduped[img] = boxes
print(f"dupes removed: {dup} | broken: {broken} | left: {len(deduped)}")

def show(cnt, title):
    print(f"\n{title}")
    for i, n in enumerate(CLASSES):
        flag = "  <-- under MIN" if cnt[i] < MIN_BOXES else ""
        print(f"{i:2d} {n:14s} {cnt[i]:6d}{flag}")
    nz = [cnt[i] for i in range(len(CLASSES)) if cnt[i] > 0]
    print(f"largest:smallest = {max(nz) / min(nz):.1f} : 1")

total = Counter(c for b in deduped.values() for c, *_ in b)
show(total, "BEFORE cap")

# rare-class images first, then accept an image only if one of its classes still needs boxes
items = list(deduped.items())
random.Random(SEED).shuffle(items)
items.sort(key=lambda kv: min(total[c] for c, *_ in kv[1]))
count, keep = Counter(), []
for img, boxes in items:
    if any(count[c] < CAP for c, *_ in boxes):
        keep.append((img, boxes))
        count.update(c for c, *_ in boxes)
show(count, f"AFTER cap ({len(keep)} images)")

dupes removed: 20 | broken: 0 | left: 7286

BEFORE cap
 0 shelf            3053
 1 wardrobe         1465
 2 dresser           855
 3 refrigerator      574
 4 tv               1128
 5 ceiling_fan       468
 6 ceiling_light       0  <-- under MIN
 7 wall_art         4508
 8 window           3891
 9 mirror            783
10 table            3983
11 bed               572
12 sofa             3160
13 door             1091
largest:smallest = 9.6 : 1

AFTER cap (4231 images)
 0 shelf            1000
 1 wardrobe         1113
 2 dresser           855
 3 refrigerator      574
 4 tv               1028
 5 ceiling_fan       468
 6 ceiling_light       0  <-- under MIN
 7 wall_art         1405
 8 window           1750
 9 mirror            783
10 table            1846
11 bed               572
12 sofa             1014
13 door             1053
largest:smallest = 3.9 : 1


In [ ]:
from tqdm.auto import tqdm

groups = defaultdict(list)   # group by rarest class in image -> each class spread across splits
for img, boxes in keep:
    groups[min((c for c, *_ in boxes), key=lambda c: count[c])].append((img, boxes))

splits = {"train": [], "val": [], "test": []}
for g in groups.values():
    random.Random(SEED).shuffle(g)
    for i, it in enumerate(g):
        k = i % 10
        splits["train" if k < 8 else "val" if k == 8 else "test"].append(it)

if OUT.exists():
    shutil.rmtree(OUT)
for split, items in splits.items():
    (OUT / "images" / split).mkdir(parents=True)
    (OUT / "labels" / split).mkdir(parents=True)
    for k, (img, boxes) in enumerate(tqdm(items, desc=split)):
        stem = f"{split}_{k:06d}"
        im = ImageOps.exif_transpose(Image.open(img)).convert("RGB")
        im.thumbnail((MAX_SIDE, MAX_SIDE))   # labels are normalized -> unaffected
        im.save(OUT / "images" / split / f"{stem}.jpg", quality=92)
        with open(OUT / "labels" / split / f"{stem}.txt", "w") as f:
            f.writelines(f"{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n" for c, x, y, w, h in boxes)
    show(Counter(c for _, b in items for c, *_ in b), f"{split}: {len(items)} images")

yaml_path = OUT / "ready_room.yaml"
yaml_path.write_text(
    f"path: {OUT}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n"
    + "".join(f"  {i}: {n}\n" for i, n in enumerate(CLASSES))
)
print(yaml_path.read_text())
!du -sh {OUT}

train:   0%|          | 0/3393 [00:00<?, ?it/s]


train: 3393 images
 0 shelf             838
 1 wardrobe          906
 2 dresser           677
 3 refrigerator      462
 4 tv                822
 5 ceiling_fan       377
 6 ceiling_light       0  <-- under MIN
 7 wall_art         1109
 8 window           1468
 9 mirror            620
10 table            1484
11 bed               460
12 sofa              816
13 door              853
largest:smallest = 3.9 : 1


val:   0%|          | 0/419 [00:00<?, ?it/s]


val: 419 images
 0 shelf              81  <-- under MIN
 1 wardrobe          109  <-- under MIN
 2 dresser            97  <-- under MIN
 3 refrigerator       57  <-- under MIN
 4 tv                 98  <-- under MIN
 5 ceiling_fan        46  <-- under MIN
 6 ceiling_light       0  <-- under MIN
 7 wall_art          162  <-- under MIN
 8 window            143  <-- under MIN
 9 mirror             86  <-- under MIN
10 table             182  <-- under MIN
11 bed                54  <-- under MIN
12 sofa              100  <-- under MIN
13 door              102  <-- under MIN
largest:smallest = 4.0 : 1


test:   0%|          | 0/419 [00:00<?, ?it/s]


test: 419 images
 0 shelf              81  <-- under MIN
 1 wardrobe           98  <-- under MIN
 2 dresser            81  <-- under MIN
 3 refrigerator       55  <-- under MIN
 4 tv                108  <-- under MIN
 5 ceiling_fan        45  <-- under MIN
 6 ceiling_light       0  <-- under MIN
 7 wall_art          134  <-- under MIN
 8 window            139  <-- under MIN
 9 mirror             77  <-- under MIN
10 table             180  <-- under MIN
11 bed                58  <-- under MIN
12 sofa               98  <-- under MIN
13 door               98  <-- under MIN
largest:smallest = 4.0 : 1
path: /content/ready_room
train: images/train
val: images/val
test: images/test
names:
  0: shelf
  1: wardrobe
  2: dresser
  3: refrigerator
  4: tv
  5: ceiling_fan
  6: ceiling_light
  7: wall_art
  8: window
  9: mirror
  10: table
  11: bed
  12: sofa
  13: door

666M	/content/ready_room


In [ ]:
from google.colab import files
!cd /content && zip -qr ready_room_dataset.zip ready_room
!ls -lh /content/ready_room_dataset.zip
files.download("/content/ready_room_dataset.zip")

-rw-r--r-- 1 root root 637M Nov 20 11:24 /content/ready_room_dataset.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# !pip install -q -U ultralytics
# from google.colab import files
# up = files.upload()   # pick ready_room_dataset.zip
# !cd /content && unzip -q ready_room_dataset.zip && rm ready_room_dataset.zip
# !ls /content/ready_room

In [ ]:
for n in fo.list_datasets():
    fo.delete_dataset(n)
shutil.rmtree(Path.home() / "fiftyone", ignore_errors=True)   # raw OI images + big CSVs
shutil.rmtree(ho["path"], ignore_errors=True)                  # raw HomeObjects
!df -h /content | tail -1

In [ ]:
!yolo detect train model=yolo26n.pt data=/content/ready_room/ready_room.yaml epochs=150 imgsz=640 batch=32 patience=30 seed=42

In [ ]:
import glob, os
from pathlib import Path
from ultralytics import YOLO

DATA = "/content/ready_room/ready_room.yaml"
BEST = max(glob.glob("/content/runs/detect/*/weights/best.pt"), key=os.path.getmtime)
print("using", BEST)
model = YOLO(BEST)
names = model.names

GROUP = {"shelf": "Topple", "wardrobe": "Topple", "dresser": "Topple", "refrigerator": "Topple",
         "tv": "Topple", "ceiling_fan": "Fall", "ceiling_light": "Fall", "wall_art": "Fall",
         "window": "Glass", "mirror": "Glass", "table": "Cover", "bed": "Zone", "sofa": "Zone",
         "door": "Exit"}
HAZARD = {"Topple", "Fall", "Glass"}

val_m = model.val(data=DATA, split="val", imgsz=640, batch=16, plots=False, verbose=False)
test_m = model.val(data=DATA, split="test", imgsz=640, batch=16, plots=True,
                   project="/content/eval", name="test", exist_ok=True, verbose=False)

print(f"\n{'class':14s}{'group':8s}{'P':>6s}{'R':>6s}{'mAP50':>7s}{'50-95':>7s}  verdict")
seen = set()
for k, ci in enumerate(test_m.box.ap_class_index):
    p, r, ap50, ap = test_m.box.class_result(k)
    n = names[int(ci)]; g = GROUP[n]; seen.add(n)
    target = 0.6 if g in HAZARD else 0.5
    verdict = "OK" if ap50 >= target else ("WEAK" if ap50 < 0.5 else "borderline")
    print(f"{n:14s}{g:8s}{p:6.2f}{r:6.2f}{ap50:7.2f}{ap:7.2f}  {verdict}")
print("not in test split:", [n for n in names.values() if n not in seen])
print(f"\nmAP50  val {val_m.box.map50:.3f} | test {test_m.box.map50:.3f}")
print(f"mAP50-95 val {val_m.box.map:.3f} | test {test_m.box.map:.3f}")

In [ ]:
from IPython.display import Image, display

cm = test_m.confusion_matrix.matrix      # rows = predicted, cols = true, last index = background
bg = cm.shape[0] - 1
idx = {n: i for i, n in names.items()}

print("Confusion pairs (share of TRUE objects predicted as the other class):")
for a, b in [("mirror", "window"), ("wardrobe", "door"), ("shelf", "dresser")]:
    for t, p in [(a, b), (b, a)]:
        ti, pi = idx[t], idx[p]
        tot = cm[:, ti].sum()
        print(f"  true {t:9s} -> {p:9s}: {cm[pi, ti]:4.0f}/{tot:4.0f} ({cm[pi, ti] / max(tot, 1):.0%})")

print("\nMissed entirely (true object, no detection):")
for n, i in idx.items():
    tot = cm[:, i].sum()
    flag = "  <-- hazard miss high" if GROUP[n] in HAZARD and cm[bg, i] / max(tot, 1) > 0.3 else ""
    print(f"  {n:14s} {cm[bg, i] / max(tot, 1):.0%}{flag}")

display(Image(f"{test_m.save_dir}/confusion_matrix_normalized.png", width=850))

In [ ]:
import numpy as np

f1 = val_m.box.f1_curve.mean(0)
rec = val_m.box.r_curve.mean(0)
px = val_m.box.px
i = int(f1.argmax())
CONF = float(round(px[i], 2))
print(f"F1-best conf = {CONF}  (mean F1 {f1[i]:.2f}, mean recall {rec[i]:.2f})")
lo = int(np.abs(px - max(CONF - 0.1, 0.05)).argmin())
print(f"recall-leaning conf = {px[lo]:.2f}  (mean F1 {f1[lo]:.2f}, mean recall {rec[lo]:.2f})")

In [ ]:
from google.colab import files
from PIL import Image as PImage

up = files.upload()
res = model.predict(list(up.keys()), conf=CONF, imgsz=640, verbose=False)
for r in res:
    print(Path(r.path).name, [(names[int(c)], round(float(s), 2)) for c, s in zip(r.boxes.cls, r.boxes.conf)])
    display(PImage.fromarray(r.plot()[..., ::-1]))

In [ ]:
tfl = YOLO(BEST).export(format="tflite", imgsz=640, half=True)   # -> best_float16.tflite
print(tfl)
tm = YOLO(tfl).val(data=DATA, split="test", imgsz=640, batch=1, verbose=False)
print(f"test mAP50  PyTorch {test_m.box.map50:.3f} | TFLite fp16 {tm.box.map50:.3f}")

In [ ]:
ml = YOLO(BEST).export(format="coreml", imgsz=640, half=True)   # -> best.mlpackage
print(ml)

In [ ]:
import tensorflow as tf

interp = tf.lite.Interpreter(model_path=str(tfl)); interp.allocate_tensors()
inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
print("input ", inp["shape"], inp["dtype"])
print("output", out["shape"], out["dtype"])

img_path = sorted(glob.glob("/content/ready_room/images/test/*.jpg"))[0]
im = PImage.open(img_path).convert("RGB")
s = 640 / max(im.size); nw, nh = round(im.width * s), round(im.height * s)
padx, pady = (640 - nw) // 2, (640 - nh) // 2
canvas = PImage.new("RGB", (640, 640), (114, 114, 114))
canvas.paste(im.resize((nw, nh), PImage.BILINEAR), (padx, pady))
x = np.asarray(canvas, np.float32)[None] / 255.0          # NHWC, RGB, 0-1

interp.set_tensor(inp["index"], x); interp.invoke()
y = interp.get_tensor(out["index"])[0]
y = y[y[:, 4] > CONF]
b = y[:, :4].copy()
normalized = len(b) > 0 and b.max() <= 1.01
if normalized: b *= 640
b[:, [0, 2]] = (b[:, [0, 2]] - padx) / s                  # undo letterbox -> original pixels
b[:, [1, 3]] = (b[:, [1, 3]] - pady) / s
print("coords normalized (0-1)?", normalized)

print("\nTFLite :", [(names[int(c)], round(float(cf), 2), bb.round().tolist()) for bb, cf, c in zip(b, y[:, 4], y[:, 5])])
pt = model.predict(img_path, conf=CONF, verbose=False)[0].boxes
print("PyTorch:", [(names[int(c)], round(float(cf), 2), bb.round().tolist()) for bb, cf, c in zip(pt.xyxy.cpu().numpy(), pt.conf, pt.cls)])

In [ ]:
import shutil, json
EXP = Path("/content/export"); shutil.rmtree(EXP, ignore_errors=True); EXP.mkdir()
shutil.copy(BEST, EXP / "best.pt")
shutil.copy(tfl, EXP / "ready_room_fp16.tflite")
shutil.copytree(ml, EXP / "ReadyRoom.mlpackage")
(EXP / "labels.txt").write_text("\n".join(names[i] for i in range(len(names))))
(EXP / "model_info.json").write_text(json.dumps({
    "input": [1, 640, 640, 3], "input_format": "RGB float 0-1, letterbox pad 114",
    "output_row": "[x1, y1, x2, y2, conf, class_id]", "coords_normalized": bool(normalized),
    "conf_threshold": CONF, "classes": [names[i] for i in range(len(names))],
    "groups": GROUP}, indent=2))
!cd /content && zip -qr ready_room_export.zip export
files.download("/content/ready_room_export.zip")